In [1]:
import numpy as np

def generate_correlated_multilayer_graph(n_nodes=100, n_layers=3, edge_prob=0.1, inter_layer_corr=0.2, save_path='synthetic_multilayer_graph.npy', seed=42):
    """
    生成带有层间相关性的多层图邻接张量
    :param n_nodes: 节点数量
    :param n_layers: 图层数量
    :param edge_prob: 每层边的概率
    :param inter_layer_corr: 图层间的边相关性（0=no corr, 1=完全相关）
    :param save_path: 保存的npy文件名
    :param seed: 随机种子
    """
    np.random.seed(seed)
    A = np.zeros((n_nodes, n_nodes, n_layers), dtype=np.float32)
    
    # 生成基础概率矩阵（用于相关性建模）
    base_edge_prob = np.random.rand(n_nodes, n_nodes)
    base_edge_prob = np.triu(base_edge_prob, 1)
    base_edge_prob = base_edge_prob + base_edge_prob.T
    # 保证对称性
    
    for m in range(n_layers):
        # 对每一层引入相关性
        noise = np.random.randn(n_nodes, n_nodes) * np.sqrt(1 - inter_layer_corr)
        noise = np.triu(noise, 1)
        noise = noise + noise.T
        correlated_prob = inter_layer_corr * base_edge_prob + (1 - inter_layer_corr) * np.random.rand(n_nodes, n_nodes)
        layer_edges = (correlated_prob < edge_prob).astype(np.float32)
        # 去除自环（对角线为0）
        np.fill_diagonal(layer_edges, 0)
        A[:, :, m] = layer_edges

    # 保证对称性
    for m in range(n_layers):
        A[:, :, m] = np.triu(A[:, :, m], 1)
        A[:, :, m] = A[:, :, m] + A[:, :, m].T

    np.save(save_path, A)
    print(f"Synthetic multilayer graph saved to {save_path}")

if __name__ == '__main__':
    generate_correlated_multilayer_graph()

Synthetic multilayer graph saved to synthetic_multilayer_graph.npy
